# Miniconda and FastGen Environment Setup

This notebook documents the steps to verify conda, install Miniconda if needed, create the `fastgen` environment, and install FastGen.

## 1. Verify Python, pip, and conda availability

In [1]:
!which python3 && which pip && conda --version 2>&1 || echo "conda not found"

/opt/venv/bin/python3
/usr/local/bin/pip
/bin/bash: line 1: conda: command not found
conda not found


## 2. Install Miniconda (only if conda is not already installed)

If `conda` is missing, run the commands below to download and install Miniconda. This step is optional once Miniconda is already installed.

In [ ]:
# Optional: install Miniconda if conda is not available
# !curl -sL https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -o /tmp/miniconda.sh
# !bash /tmp/miniconda.sh -b -p $HOME/miniconda3
# !bash -lc "source $HOME/.bashrc && conda --version"

## 3. Create the `fastgen` conda environment

In [2]:
!conda create -y -n fastgen python=3.12.3 pip

/bin/bash: line 1: conda: command not found


In [3]:
%%bash
source ~/.bashrc
conda activate fastgen
python --version


bash: line 2: conda: command not found


Python 3.12.3


In [4]:
%%bash
if [ -d .git ]; then
  echo "Installing from current FastGen repository"
  pip install -e .
else
  git clone https://github.com/NVlabs/FastGen.git repo_clone
  cd repo_clone
  pip install -e .
fi


Installing from current FastGen repository
Obtaining file:///workspace/user_homes/hseth/FastGen


  DEPRECATION: Setting PIP_CONSTRAINT will not affect build constraints in the future, pip 26.2 will enforce this behaviour change. A possible replacement is to specify build constraints using --build-constraint or PIP_BUILD_CONSTRAINT. To disable this warning without any build constraints set --use-feature=build-constraint or PIP_USE_FEATURE="build-constraint".


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/4.1 MB ? eta -:--:--━━━━━━━━━━━━━━━━━ 0.8/4.1 MB 5.7 MB/s eta 0:00:01━━━━━━━━━━━ 4.1/4.1 MB 17.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/10.0 MB ? eta -:--:--━━━━━━━━━━━ 10.0/10.0 MB 115.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/3.1 MB ? eta -:--:

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorrt-llm 1.2.0 requires apache-tvm-ffi==0.1.6, but you have apache-tvm-ffi 0.1.11 which is incompatible.
tensorrt-llm 1.2.0 requires onnx<1.20.0,>=1.18.0, but you have onnx 1.21.0 which is incompatible.
tensorrt-llm 1.2.0 requires setuptools<80, but you have setuptools 82.0.1 which is incompatible.
tensorrt-llm 1.2.0 requires transformers==4.57.3, but you have transformers 4.49.0 which is incompatible.
vllm 0.17.2.dev0+g95c0f928c.d20260313.cu131 requires llguidance<1.4.0,>=1.3.0; platform_machine == "x86_64" or platform_machine == "arm64" or platform_machine == "aarch64" or platform_machine == "s390x" or platform_machine == "ppc64le", but you have llguidance 0.7.29 which is incompatible.
vllm 0.17.2.dev0+g95c0f928c.d20260313.cu131 requires mistral_common[image]>=1.9.1, but you have mistral-common 1.8.6 which i

## 6. Credentials (Optional)

For W&B logging, [get your API key](https://wandb.ai/settings) and save it to `credentials/wandb_api.txt` or set the `WANDB_API_KEY` environment variable.
Without either of these, W&B will prompt for your API key interactively.

For more details, including S3 storage and other environment variables, see [fastgen/configs/README.md](fastgen/configs/README.md#environment-variables).

In [5]:
!python scripts/download_data.py --dataset cifar10

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Multiple distributions found for package modelopt. Picked distribution: nvidia-modelopt
Error.  nthreads cannot be larger than environment variable "NUMEXPR_MAX_THREADS" (64)xFormers not available
[Jun 12, 2026 - 10:03:46 | INFO | __main__:main:623 ] FastGen Dataset Setup: cifar10
[Jun 12, 2026 - 10:03:46 | INFO | __main__:main:624 ] Data directory:       /workspace/user_homes/hseth/FastGen/FASTGEN_OUTPUT/DATA
[Jun 12, 2026 - 10:03:46 | INFO | __main__:main:625 ] Checkpoint directory: /workspace/user_homes/hseth/FastGen/FASTGEN_OUTPUT/MODEL
[Jun 12, 2026 - 10:03:46 | INFO | __main__:clone_repo:195 ] Cloning edm repo to /tmp/tmpwxw_2ac9/edm...
[Jun 12, 2026 - 10:03:48 | INFO 

### Basic Training

If you run out-of-memory, try a smaller batch-size, e.g., `dataloader_train.batch_size=32`, which automatically uses gradient accumulation to match the global batch-size.

**Expected Output:** See the training log for a link to the run on [wandb.ai](https://wandb.ai). Training outputs go to `$FASTGEN_OUTPUT_ROOT/{project}/{group}/{name}/`. With default settings, outputs are organized as follows:
```
FASTGEN_OUTPUT/fastgen/cifar10/debug/
├── checkpoints/    # Model checkpoints in the format {iteration:07d}.pth
│   ├── 0001000.pth
│   └── ...
├── config.yaml     # Resolved configuration for reproducibility
├── wandb_id.txt    # W&B run ID for resuming
└── ...          
```

In [14]:
!python train.py --config=fastgen/configs/experiments/CosmosPredict2/config_dmd2.py - log_config.name=cosmos_predict2_dmd2

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Error.  nthreads cannot be larger than environment variable "NUMEXPR_MAX_THREADS" (64)Multiple distributions found for package modelopt. Picked distribution: nvidia-modelopt
xFormers not available
[Jun 12, 2026 - 12:00:21 | INFO | fastgen.configs.config_utils:serialize_config:315 ] Config is saved at FASTGEN_OUTPUT/fastgen/cosmos_predict2_dmd2/cosmos_predict2_dmd2/config.yaml
[Jun 12, 2026 - 12:00:21 | INFO | fastgen.utils.scripts:setup:87 ] No DDP or FSDP parallelism
[Jun 12, 2026 - 12:00:21 | CRITICAL | fastgen.utils.scripts:setup:118 ] Global batch size: 1 (Batch size per GPU: 1, Gradient accumulation rounds: 1, World size: 1)
[Jun 12, 2026 - 12:00:21 | CRITICAL | fastgen

In [ ]:
%%bash
source ~/.bashrc
conda activate fastgen
python scripts/inference/image_model_inference.py --config fastgen/configs/experiments/EDM/config_dmd2_test.py \
  --classes=10 --prompt_file=scripts/inference/prompts/classes.txt --ckpt_path=FASTGEN_OUTPUT/fastgen/cifar10/debug/checkpoints/0002000.pth - log_config.name=test_inference


bash: line 2: conda: command not found
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Error.  nthreads cannot be larger than environment variable "NUMEXPR_MAX_THREADS" (64)Multiple distributions found for package modelopt. Picked distribution: nvidia-modelopt
xFormers not available


[Jun 12, 2026 - 12:07:25 | INFO | fastgen.configs.config_utils:serialize_config:315 ] Config is saved at FASTGEN_OUTPUT/fastgen/cifar10/test_inference/samples/config.yaml
[Jun 12, 2026 - 12:07:25 | INFO | fastgen.utils.scripts:setup:87 ] No DDP or FSDP parallelism
[Jun 12, 2026 - 12:07:25 | CRITICAL | fastgen.utils.scripts:setup:118 ] Global batch size: 64 (Batch size per GPU: 64, Gradient accumulation rounds: 1, World size: 1)
[Jun 12, 2026 - 12:07:25 | CRITICAL | fastgen.utils.scripts:set_cuda_backend:46 ] cuDNN deterministic: False, cuDNN benchmark: True, enable TF32: True
[Jun 12, 2026 - 12:07:25 | INFO | scripts.inference.inference_utils:load_prompts:71 ] Loaded 10 prompts from /workspace/user_homes/hseth/FastGen/scripts/inference/prompts/classes.txt
[Jun 12, 2026 - 12:07:25 | INFO | fastgen.utils.basic_utils:set_random_seed:144 ] Using random seed 0.
[Jun 12, 2026 - 12:07:25 | CRITICAL | fastgen.methods.model:set_precision:172 ] Model and data precision: torch.float32. AMP traini

In [4]:
%%bash
source ~/.bashrc
conda activate fastgen
python scripts/fid/compute_fid_from_ckpts.py \
    --config fastgen/configs/experiments/EDM/config_dmd2_cifar10.py 

bash: line 2: conda: command not found


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Error.  nthreads cannot be larger than environment variable "NUMEXPR_MAX_THREADS" (64)Multiple distributions found for package modelopt. Picked distribution: nvidia-modelopt
xFormers not available


[Jun 12, 2026 - 12:14:35 | INFO | fastgen.configs.config_utils:serialize_config:315 ] Config is saved at FASTGEN_OUTPUT/fastgen/cifar10/debug/samples/config.yaml
[Jun 12, 2026 - 12:14:35 | INFO | fastgen.utils.scripts:setup:87 ] No DDP or FSDP parallelism
[Jun 12, 2026 - 12:14:35 | INFO | fastgen.utils.scripts:setup:113 ] Changing gradient accumulation rounds from 1 to 8 to match requested global batch size.
[Jun 12, 2026 - 12:14:35 | CRITICAL | fastgen.utils.scripts:setup:118 ] Global batch size: 2048 (Batch size per GPU: 256, Gradient accumulation rounds: 8, World size: 1)
[Jun 12, 2026 - 12:14:35 | CRITICAL | fastgen.utils.scripts:set_cuda_backend:46 ] cuDNN deterministic: False, cuDNN benchmark: True, enable TF32: True
[Jun 12, 2026 - 12:14:35 | INFO | fastgen.utils.basic_utils:set_random_seed:144 ] Using random seed 0.
[Jun 12, 2026 - 12:14:35 | CRITICAL | fastgen.methods.model:set_precision:172 ] Model and data precision: torch.float32. AMP training precision: None. AMP en-/decod

100%|██████████| 196/196 [02:18<00:00,  1.42batch/s]


[Jun 12, 2026 - 12:16:58 | INFO | __main__:main:134 ] Loading model from "FASTGEN_OUTPUT/fastgen/cifar10/debug/checkpoints/0002000.pth"...
[Jun 12, 2026 - 12:16:58 | INFO | fastgen.utils.checkpointer:load:151 ] Loading model from FASTGEN_OUTPUT/fastgen/cifar10/debug/checkpoints/0002000.pth
[Jun 12, 2026 - 12:16:59 | INFO | fastgen.utils.checkpointer:load:154 ] Loading the model_dict...
[Jun 12, 2026 - 12:16:59 | INFO | fastgen.utils.checkpointer:load:159 ] Model net, loading info: <All keys matched successfully>
[Jun 12, 2026 - 12:16:59 | WARNING | fastgen.utils.checkpointer:load:161 ] Model ema_9999 not found in checkpoint.
[Jun 12, 2026 - 12:16:59 | WARNING | fastgen.utils.checkpointer:load:161 ] Model ema_99995 not found in checkpoint.
[Jun 12, 2026 - 12:16:59 | WARNING | fastgen.utils.checkpointer:load:161 ] Model ema_9996 not found in checkpoint.
[Jun 12, 2026 - 12:17:00 | INFO | fastgen.utils.checkpointer:load:159 ] Model fake_score, loading info: <All keys matched successfully>


...

[Jun 12, 2026 - 12:23:49 | INFO | __main__:main:134 ] Loading model from "FASTGEN_OUTPUT/fastgen/cifar10/debug/checkpoints/0005000.pth"...
[Jun 12, 2026 - 12:23:49 | INFO | fastgen.utils.checkpointer:load:151 ] Loading model from FASTGEN_OUTPUT/fastgen/cifar10/debug/checkpoints/0005000.pth
[Jun 12, 2026 - 12:23:50 | INFO | fastgen.utils.checkpointer:load:154 ] Loading the model_dict...
[Jun 12, 2026 - 12:23:50 | INFO | fastgen.utils.checkpointer:load:159 ] Model net, loading info: <All keys matched successfully>
[Jun 12, 2026 - 12:23:50 | WARNING | fastgen.utils.checkpointer:load:161 ] Model ema_9999 not found in checkpoint.
[Jun 12, 2026 - 12:23:50 | WARNING | fastgen.utils.checkpointer:load:161 ] Model ema_99995 not found in checkpoint.
[Jun 12, 2026 - 12:23:50 | WARNING | fastgen.utils.checkpointer:load:161 ] Model ema_9996 not found in checkpoint.
[Jun 12, 2026 - 12:23:50 | INFO | fastgen.utils.checkpointer:load:159 ] Model fake_score, loading info: <All keys matched successfully>


100%|██████████| 196/196 [02:16<00:00,  1.44batch/s]


[Jun 12, 2026 - 12:26:16 | INFO | scripts.fid.fid:calc:114 ] Loading dataset reference statistics from "FASTGEN_OUTPUT/DATA/fid-refs/cifar10-32x32.npz"...


Traceback (most recent call last):
  File "/workspace/user_homes/hseth/FastGen/scripts/fid/compute_fid_from_ckpts.py", line 249, in <module>
    main(config)
  File "/workspace/user_homes/hseth/FastGen/scripts/fid/compute_fid_from_ckpts.py", line 212, in main
    calc(
  File "/workspace/user_homes/hseth/FastGen/scripts/fid/fid.py", line 117, in calc
    with open_url(ref_path) as f:
         ^^^^^^^^^^^^^^^^^^
  File "/workspace/user_homes/hseth/FastGen/fastgen/utils/io_utils.py", line 128, in open_url
    return url if return_filename else open(url, "rb")
                                       ^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'FASTGEN_OUTPUT/DATA/fid-refs/cifar10-32x32.npz'


CalledProcessError: Command 'b'source ~/.bashrc\nconda activate fastgen\npython scripts/fid/compute_fid_from_ckpts.py \\\n    --config fastgen/configs/experiments/EDM/config_dmd2_cifar10.py \n'' returned non-zero exit status 1.

## 8. Documentation

Detailed documentation is available in each component's README:

| Component | Documentation | Description |
|-----------|---------------|-------------|
| **Methods** | [fastgen/methods/README.md](fastgen/methods/README.md) | Training methods (sCM, MeanFlow, DMD2, Self-Forcing, etc.) |
| **Networks** | [fastgen/networks/README.md](fastgen/networks/README.md) | Network architectures (EDM, SD, SDXL, Flux, Qwen-Image, WAN, CogVideoX, Cosmos) and pretrained models |
| **Configs** | [fastgen/configs/README.md](fastgen/configs/README.md) | Configuration system, environment variables, and creating custom configs |
| **Datasets** | [fastgen/datasets/README.md](fastgen/datasets/README.md) | Dataset preparation and WebDataset loaders |
| **Callbacks** | [fastgen/callbacks/README.md](fastgen/callbacks/README.md) | Training callbacks (EMA, logging, gradient clipping, etc.) |
| **Inference** | [scripts/README.md](scripts/README.md) | Inference modes (T2I, T2V, I2V, V2V, etc.) and FID evaluation |
| **Third Party** | [fastgen/third_party/README.md](fastgen/third_party/README.md) | Third-party dependencies (Depth Anything V2, etc.) |